# Kaggle — Light-CNN + LFCC | ASVspoof 2019 LA

In [ ]:
import subprocess
subprocess.run(["pip", "install", "soundfile", "librosa", "-q"])
import os, glob, numpy as np, pandas as pd, soundfile as sf, librosa
import scipy.fftpack as fft_
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_curve, roc_auc_score
import time, json, math

def find_la_root():
    candidates = ["/kaggle/input/asvpoof-2019-dataset/LA"]
    for m in glob.glob("/kaggle/input/**/ASVspoof2019_LA_train", recursive=True):
        candidates.insert(0, os.path.dirname(m))
    for c in candidates:
        if os.path.isdir(c) and os.path.isdir(os.path.join(c, "ASVspoof2019_LA_train")):
            return c
    raise RuntimeError("LA root not found. Add: https://www.kaggle.com/datasets/awsaf49/asvpoof-2019-dataset")

LA_ROOT   = find_la_root()
PROTO_DIR = os.path.join(LA_ROOT, "ASVspoof2019_LA_cm_protocols")
FLAC_DIRS = {
    "train": os.path.join(LA_ROOT, "ASVspoof2019_LA_train", "flac"),
    "dev"  : os.path.join(LA_ROOT, "ASVspoof2019_LA_dev",   "flac"),
    "eval" : os.path.join(LA_ROOT, "ASVspoof2019_LA_eval",  "flac"),
}
PROTO_FILES = {
    "train": os.path.join(PROTO_DIR, "ASVspoof2019.LA.cm.train.trn.txt"),
    "dev"  : os.path.join(PROTO_DIR, "ASVspoof2019.LA.cm.dev.trl.txt"),
    "eval" : os.path.join(PROTO_DIR, "ASVspoof2019.LA.cm.eval.trl.txt"),
}
print(f"LA root: {LA_ROOT}")
for k, p in {**FLAC_DIRS, **PROTO_FILES}.items():
    cnt = len(os.listdir(p)) if os.path.isdir(p) else ("-" if os.path.isfile(p) else "MISSING")
    status = "OK" if os.path.exists(p) else "MISSING"
    print(f"  {status} [{cnt}]  {k}")


In [ ]:
rows = []
for part, proto_path in PROTO_FILES.items():
    with open(proto_path, encoding="utf-8") as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 5:
                continue
            rows.append({
                "speaker_id": p[0],
                "audio_id"  : p[1],
                "attack_id" : p[3],
                "key"       : p[4],
                "is_spoof"  : int(p[4] == "spoof"),
                "partition" : part,
                "file_path" : os.path.join(FLAC_DIRS[part], p[1] + ".flac"),
            })
manifest = pd.DataFrame(rows)
print(f"Total rows: {len(manifest)}")
for part in ["train", "dev", "eval"]:
    s   = manifest[manifest["partition"] == part]
    bon = len(s[s["key"] == "bonafide"])
    spf = len(s[s["key"] == "spoof"])
    print(f"  {part}: {len(s)} | bon={bon} spoof={spf} ratio={spf/max(bon,1):.1f}:1")


In [ ]:
def load_audio(path, sr=16000):
    y, orig = sf.read(path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    if orig != sr:
        y = librosa.resample(y, orig_sr=orig, target_sr=sr)
    return y

def process(y, train=False, alpha=0.97, T=64000, top_db=40):
    y  = np.concatenate([[y[0]], y[1:] - alpha * y[:-1]])
    iv = librosa.effects.split(y=y, top_db=top_db)
    if len(iv):
        t = np.concatenate([y[s:e] for s, e in iv])
        if len(t) > 1000:
            y = t
    n = len(y)
    if n >= T:
        st = np.random.randint(0, n - T + 1) if train else (n - T) // 2
        y  = y[st:st + T]
    else:
        y = np.pad(y, (0, T - n), mode="wrap")
    return y / (np.max(np.abs(y)) + 1e-7)

def extract_lfcc(y, sr=16000, n_fft=1024, hop=256, n_ceps=20, T=251):
    S      = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop, center=True)) ** 2
    n_bins = S.shape[0]
    fb     = np.zeros((n_ceps, n_bins), dtype=np.float32)
    pts    = np.linspace(0, n_bins - 1, n_ceps + 2, dtype=int)
    for i in range(n_ceps):
        lo, mid, hi = pts[i], pts[i + 1], pts[i + 2]
        if mid > lo:  fb[i, lo:mid] = np.linspace(0, 1, mid - lo)
        if hi  > mid: fb[i, mid:hi] = np.linspace(1, 0, hi - mid)
    log_e  = np.log(np.maximum(fb @ S, 1e-8))
    static = fft_.dct(log_e, type=2, axis=0, norm="ortho")[:n_ceps]
    d1     = librosa.feature.delta(static, order=1)
    d2     = librosa.feature.delta(static, order=2)
    feat   = np.vstack([static, d1, d2]).astype(np.float32)
    return feat[:, :T] if feat.shape[1] >= T else np.pad(feat, ((0, 0), (0, T - feat.shape[1])), mode="edge")

def extract_mel(y, sr=16000, n_fft=1024, hop=256, n_mels=128, fmin=20, fmax=8000, T=251):
    M  = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop,
                                          n_mels=n_mels, fmin=fmin, fmax=fmax)
    lM = librosa.power_to_db(M, ref=np.max).astype(np.float32)
    return lM[:, :T] if lM.shape[1] >= T else np.pad(lM, ((0, 0), (0, T - lM.shape[1])), mode="edge")

_r  = manifest[manifest["partition"] == "train"].iloc[0]
_y  = process(load_audio(_r["file_path"]))
_lf = extract_lfcc(_y)
_ml = extract_mel(_y)
assert _lf.shape == (60, 251), f"LFCC shape error: {_lf.shape}"
assert _ml.shape == (128, 251), f"Mel shape error: {_ml.shape}"
print(f"LFCC: {_lf.shape} | Mel: {_ml.shape}  OK")
del _y, _lf, _ml, _r


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, a=0.75, g=2.0, ls=0.05):
        super().__init__()
        self.a, self.g, self.ls = a, g, ls
    def forward(self, inp, tgt):
        ce = F.cross_entropy(inp, tgt, reduction="none", label_smoothing=self.ls)
        pt = torch.exp(-ce)
        at = torch.where(tgt == 1, self.a, 1.0 - self.a)
        return (at * (1 - pt) ** self.g * ce).mean()

def compute_eer(yt, ys):
    fpr, tpr, _ = roc_curve(yt, ys, pos_label=1)
    fnr = 1 - tpr
    i   = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[i] + fnr[i]) / 2)

def make_sampler(labels):
    cc = np.bincount(labels)
    sw = torch.FloatTensor((1.0 / cc)[labels])
    return WeightedRandomSampler(sw, len(sw), replacement=True)

def run_eval(model, loader, device, amp):
    model.eval()
    probs, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            with torch.amp.autocast(device_type=device.type, enabled=amp):
                p = torch.softmax(model(xb.to(device)), dim=1)[:, 1].cpu().numpy()
            probs.append(p)
            targets.append(yb.numpy())
    yp = np.concatenate(probs)
    yt = np.concatenate(targets)
    return compute_eer(yt, yp), roc_auc_score(yt, yp)

def run_training(model, tl, dl, tds, crit, opt, sched, scaler, name, epochs, device, amp):
    best_eer, best_auc = float("inf"), 0.0
    history = []
    spath = f"/kaggle/working/{name}_best.pth"
    os.makedirs("/kaggle/working", exist_ok=True)
    for ep in range(1, epochs + 1):
        model.train()
        tloss, t0 = 0.0, time.time()
        for xb, yb in tl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            tloss += loss.item() * xb.size(0)
        tloss /= len(tds)
        sched.step(ep)
        e, auc = run_eval(model, dl, device, amp)
        elapsed = time.time() - t0
        print(f"Ep {ep:02d}/{epochs} | loss={tloss:.4f} | EER={e*100:.2f}% | AUC={auc:.4f} | {elapsed:.0f}s")
        history.append({"epoch": ep, "train_loss": round(tloss, 6),
                         "val_eer": round(e, 6), "val_auc": round(auc, 6)})
        if e < best_eer:
            best_eer, best_auc = e, auc
            torch.save({"epoch": ep, "state_dict": model.state_dict(),
                         "eer": best_eer, "auc": best_auc}, spath)
            print(f"  >>> Best: EER={best_eer*100:.2f}% AUC={best_auc:.4f}")
    json.dump(history, open(f"/kaggle/working/{name}_history.json", "w"), indent=2)
    print(f"Done. Best EER={best_eer*100:.2f}% | Saved: {spath}")
    return history, best_eer, best_auc

def plot_hist(hist, name):
    import matplotlib.pyplot as plt
    ep   = [h["epoch"] for h in hist]
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    ax[0].plot(ep, [h["train_loss"] for h in hist], "#3498db", lw=2)
    ax[0].set(title="Loss", xlabel="Epoch")
    eers = [h["val_eer"] * 100 for h in hist]
    ax[1].plot(ep, eers, "#e74c3c", lw=2, marker="o", ms=3)
    ax[1].axhline(min(eers), color="gray", ls="--", label=f"Best: {min(eers):.2f}%")
    ax[1].legend(); ax[1].set(title="Dev EER %")
    aucs = [h["val_auc"] for h in hist]
    ax[2].plot(ep, aucs, "#2ecc71", lw=2, marker="o", ms=3)
    ax[2].axhline(max(aucs), color="gray", ls="--", label=f"Best: {max(aucs):.4f}")
    ax[2].legend(); ax[2].set(title="Dev AUC")
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/{name}_curve.png", dpi=150)
    plt.show()


In [ ]:
class MFM(nn.Module):
    def forward(self, x):
        a, b = torch.split(x, x.size(1) // 2, dim=1)
        return torch.max(a, b)

class LightCNN(nn.Module):
    def __init__(self, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,  64,  5, 1, 2),  MFM(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64,  1),         MFM(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 96,  3, 1, 1),  MFM(), nn.MaxPool2d(2, 2), nn.BatchNorm2d(48),
            nn.Conv2d(48, 96,  1),         MFM(), nn.BatchNorm2d(48),
            nn.Conv2d(48, 128, 3, 1, 1),  MFM(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 1),         MFM(), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64,  3, 1, 1),  MFM(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 64,  1),         MFM(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 64,  3, 1, 1),  MFM(), nn.MaxPool2d(2, 2),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(32, 64), nn.ReLU(), nn.Dropout(drop), nn.Linear(64, 2)
        )
    def forward(self, x): return self.head(self.net(x))


In [ ]:
class LFCCDataset(Dataset):
    def __init__(self, df, train=False):
        self.df    = df.reset_index(drop=True)
        self.train = train
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:
            y = process(load_audio(r["file_path"]), self.train)
            x = extract_lfcc(y)
        except Exception:
            x = np.zeros((60, 251), dtype=np.float32)
        if self.train:
            if np.random.rand() < 0.5:
                t  = np.random.randint(1, 31)
                t0 = np.random.randint(0, max(1, 251 - t))
                x[:, t0:t0 + t] = x.mean()
            if np.random.rand() < 0.5:
                f  = np.random.randint(1, 11)
                f0 = np.random.randint(0, max(1, 60 - f))
                x[f0:f0 + f, :] = x.mean()
        return torch.from_numpy(x).unsqueeze(0), torch.tensor(int(r["is_spoof"]), dtype=torch.long)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP    = DEVICE.type == "cuda"
print(f"Device: {DEVICE}")
if AMP:
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

tdf = manifest[manifest["partition"] == "train"].reset_index(drop=True)
ddf = manifest[manifest["partition"] == "dev"].reset_index(drop=True)
tds = LFCCDataset(tdf, train=True)
dds = LFCCDataset(ddf, train=False)
tl  = DataLoader(tds, 128, sampler=make_sampler(tdf["is_spoof"].values), num_workers=2, pin_memory=True)
dl  = DataLoader(dds, 256, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train batches: {len(tl)} | Dev batches: {len(dl)}")


In [ ]:
model = LightCNN().to(DEVICE)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"LightCNN params: {n:,}")
with torch.no_grad():
    out = model(torch.zeros(2, 1, 60, 251).to(DEVICE))
    print(f"Forward: {out.shape}  (expected [2, 2])")

crit  = FocalLoss(0.75, 2.0, 0.05)
opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2, eta_min=1e-6)
sc    = torch.amp.GradScaler(device=DEVICE.type, enabled=AMP)
hist, best_eer, best_auc = run_training(model, tl, dl, tds, crit, opt, sched, sc, "light_cnn_lfcc", 30, DEVICE, AMP)


In [ ]:
plot_hist(hist, "light_cnn_lfcc")